RAG with PDF as context source
Install the dependencies first
pip install pypdf


In [1]:
#importing libraries
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. SETUP
# ─────────────────────────────────────────

print("🔧 Loading models...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
llm = OllamaLLM(model="llama3.2:3b")
client = chromadb.Client()
collection = client.create_collection("attention_paper")
print("✅ Models ready\n")

🔧 Loading models...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1160.26it/s]


✅ Models ready



In [3]:
# 2. LOAD PDF
# ─────────────────────────────────────────

loader = PyPDFLoader("NIPS-2017-attention-is-all-you-need-Paper.pdf")  # ← your filename here
pages = loader.load()
print(f"✅ Loaded {len(pages)} pages")
print(f"📄 Sample from page 1:\n{pages[0].page_content[:300]}\n")

✅ Loaded 11 pages
📄 Sample from page 1:
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aid



In [4]:
# 3. CHUNK
# ─────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(pages)
texts = [chunk.page_content for chunk in chunks]
print(f"✅ Split into {len(texts)} chunks")
print(f"📦 Sample chunk:\n{texts[0]}\n")


✅ Split into 76 chunks
📦 Sample chunk:
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or



In [5]:
# 4. EMBED + STORE IN CHROMADB
# ─────────────────────────────────────────

print("⏳ Embedding chunks...")
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"chunk{i}" for i in range(len(texts))]
)
print(f"\n✅ Indexed {len(texts)} chunks into ChromaDB\n")

⏳ Embedding chunks...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.12it/s]


✅ Indexed 76 chunks into ChromaDB



In [6]:
# 5. RAG FUNCTION
# ─────────────────────────────────────────

def ask(question):
    print(f"❓ Question: {question}\n")

    # Embed the question
    query_embedding = embedding_model.encode(question).tolist()

    # Retrieve top 4 relevant chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=4
    )
    retrieved_chunks = results["documents"][0]

    # Build prompt
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""You are an expert AI research assistant.
Answer the question using ONLY the context provided below.
Be specific and reference details from the context.
If the answer is not in the context, say "The paper doesn't cover that."

Context:
{context}

Question: {question}

Answer:"""

    print(f"📋 Retrieved {len(retrieved_chunks)} chunks")
    print(f"🤖 Answer:")
    response = llm.invoke(prompt)
    print(response)
    print("\n" + "─"*60 + "\n")

In [7]:
# 6. ASK QUESTIONS ABOUT THE PAPER
# ─────────────────────────────────────────
ask("Who are the authors of this paper?")

❓ Question: Who are the authors of this paper?

📋 Retrieved 4 chunks
🤖 Answer:
The authors of the papers listed in the context are not explicitly stated as being the same. However, Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu co-authored "Exploring the limits of language modeling" ([13]).

────────────────────────────────────────────────────────────



In [8]:
ask("What is this paper about?")

❓ Question: What is this paper about?

📋 Retrieved 4 chunks
🤖 Answer:
This paper appears to be about the Transformer model, specifically its architecture and components such as attention and masking mechanisms. It does not provide explicit information on a specific topic or problem that the paper is addressing, but based on the context of being part of the research at Google Brain and Google Research, it can be inferred that the paper may focus on improving or exploring the Transformer model for tasks like neural machine translation.

────────────────────────────────────────────────────────────



In [9]:
ask("What BLEU scores did the model achieve on translation tasks?")

❓ Question: What BLEU scores did the model achieve on translation tasks?

📋 Retrieved 4 chunks
🤖 Answer:
According to the context, the model achieved the following BLEU scores:

* On the WMT 2014 English-to-German translation task, the big Transformer model achieved a new state-of-the-art BLEU score of 28.4.
* On the WMT 2014 English-to-French translation task, the same big Transformer model achieved a BLEU score of 41.0.

────────────────────────────────────────────────────────────



In [ ]:
ask("how many moons does Jupiter have?")
#out of scope question to test if the model is just making up answers or admitting when it doesn't know something

❓ Question: how many moons does Jupiter have?

📋 Retrieved 4 chunks
🤖 Answer:
The paper doesn't cover that. The provided context is about AI research, specifically transformer models and their training costs, and does not mention astronomy or the number of moons on Jupiter.

────────────────────────────────────────────────────────────



Why 500 for chunk_size
Chunk size is measured in characters (not tokens, not words).
The reasoning for 500 is based on two competing pressures:
Too small (e.g. 100 chars)          Too large (e.g. 2000 chars)
        ↓                                       ↓
Chunks lose context                 Chunks contain too many topics
"BLEU score was 28.4"               "intro + methodology + results +
 ← what score? for what?             conclusion all mixed together"
        ↓                                       ↓
Retrieval finds the chunk           Retrieval finds the chunk
but LLM can't answer well           but LLM gets confused by noise
500 characters is roughly 3–5 sentences — enough to contain a complete thought with surrounding context, but focused enough to be about one specific topic.
For the Attention paper specifically, most paragraphs are 2–5 sentences, so 500 characters aligns naturally with paragraph boundaries.


Why 50 for chunk_overlap
Overlap exists to solve the boundary problem:

Chunk 1 ends:   "...the model achieves 28.4 BLEU on English-
Chunk 2 starts:  to-German translation tasks, outperforming..."

Without overlap, a question about BLEU scores might retrieve chunk 2 (which has the context) but miss chunk 1 (which has the actual number). With 50 character overlap, the end of chunk 1 is repeated at the start of chunk 2 — so important information at boundaries isn't lost.
50 characters = roughly one sentence of overlap. Enough to bridge boundaries without wasting too much space on repetition.

How chunk_size Should Actually Be Decided?

In practice you tune it based on three factors:
1. Nature of your document
Document typeRecommended chunk_size:
Academic paper (dense paragraphs)500–800
News articles300–500
Legal documents800–1200
Code1000–2000
FAQ / bullet points200–300
The Attention paper has dense academic paragraphs — 500 fits well.

embedding model's token limit
all-MiniLM-L6-v2 has a 256 token limit (~1000 characters). If your chunk exceeds that, the embedding model silently truncates it — meaning the end of the chunk is never embedded and never searchable.
all-MiniLM-L6-v2 limit:  256 tokens  ≈ 1000 characters
Our chunk_size:           500 characters  ← safely within limit ✅
This is actually the most important constraint people miss.

 LLM's context window
You retrieve top 4 chunks and inject them into the prompt:
4 chunks × 500 chars = 2000 chars ≈ 500 tokens of context
+ system prompt + question         ≈ 200 tokens
─────────────────────────────────────────────────
Total input:                       ≈ 700 tokens
Llama 3.2 has an 8000 token context window — you're well within limits. But if you increased chunk_size to 2000 and retrieved 10 chunks, you'd start hitting limits.

In [11]:
# Add this after chunking to inspect
for i, text in enumerate(texts[:5]):
    print(f"--- Chunk {i} ({len(text)} chars) ---")
    print(text)
    print()

--- Chunk 0 (500 chars) ---
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or

--- Chunk 1 (467 chars) ---
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable a

 good chunk should:

✅ Make sense on its own when read in isolation
✅ Be about one main topic
✅ Not end mid-sentence (the separators list handles this)

A bad chunk looks like:

❌ "...4 on WMT 2014 English-" — cut mid-thought
❌ A chunk that's just a page header or whitespace
❌ A chunk mixing 4 completely unrelated topics


The separators List Explained
pythonseparators=["\n\n", "\n", ".", " "]
This tells the splitter the priority order for where to cut:
Try to split on \n\n first  (paragraph breaks — ideal)
    ↓ if chunk still too big
Try to split on \n           (line breaks)
    ↓ if chunk still too big
Try to split on .            (sentence endings)
    ↓ if chunk still too big
Split on spaces              (last resort)
This is why it's called RecursiveCharacterTextSplitter — it recursively tries cleaner split points before resorting to ugly ones. The result is chunks that respect natural language boundaries as much as possible.